## E-Commerce Returns & Customer Behaviour Analysis

## 1. Data Understanding

### 1.1 Orders Table

In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path

In [2]:
orders = pd.read_csv("../data/raw/eco_orders.csv")
orders_items = pd.read_csv("../data/raw/eco_order_items.txt")
deliveries = pd.read_csv("../data/raw/eco_deliveries.csv")
products = pd.read_csv("../data/raw/eco_products.csv")
customers = pd.read_csv("../data/raw/eco_customers.csv")
returns = pd.read_csv("../data/raw/eco_returns.csv")
warehouses = pd.read_csv("../data/raw/eco_warehouses.csv")


C:\Users\Akash\AppData\Local\Temp\ipykernel_27172\2331767140.py:1: DtypeWarning: Columns (8: Unnamed: 8) have mixed types. Specify dtype option on import or set low_memory=False.
  orders = pd.read_csv("../data/raw/eco_orders.csv")


## Data Understanding

In this phase, we will explore all seven datasets to understand their structure, columns, data types, and data quality.



In [59]:
data = {
    "orders": orders,
    "orders_items": orders_items,
    "deliveries": deliveries,
    "products": products,
    "customers": customers,
    "returns": returns,
    "warehouses": warehouses
}

for name, df in data.items():

    print(f"{name} info:\n")
    df.info()

    print("________________________________" * 2)
    print(f"{name} shape:\n", df.shape)

    print("________________________________" * 2)
    print(f"{name} columns:\n", df.columns.tolist())

    print("________________________________" * 2)
    print(f"{name} dtypes:\n", df.dtypes)

    print("________________________________" * 2)
    print(f"{name} null values:\n", df.isnull().sum())
    
    print("________________________________"*2)
    print(f"{name} duplicated values:\n", df.duplicated().sum())
    
    print("________________________________"*2)
    print(f"{name} unique values:\n", df.nunique())
    
    print("________________________________"*2)
    print(f"{name} describe:\n")
    print(df.describe(include="all"))

    print("________________________________"*2)
    print(f"{name} head:\n")
    print(df.head())

    print("________________________________"*2)
    print(f"{name} tail:\n")
    print(df.tail(),"\n")
    
    print("************************************************************"*2)

orders info:

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   order_id        1000000 non-null  str           
 1   customer_id     1000000 non-null  str           
 2   order_date      1000000 non-null  datetime64[us]
 3   warehouse_id    1000000 non-null  str           
 4   payment_method  1000000 non-null  str           
 5   order_status    1000000 non-null  str           
dtypes: datetime64[us](1), str(5)
memory usage: 45.8 MB
________________________________________________________________
orders shape:
 (1000000, 6)
________________________________________________________________
orders columns:
 ['order_id', 'customer_id', 'order_date', 'warehouse_id', 'payment_method', 'order_status']
________________________________________________________________
orders dtypes:
 order_id                     str
customer_id      

### Data Understanding – Cross-Checking

#### Check 1: Orders–Customers Cross-Check

**Objective:** To verify whether all customer IDs in the Orders table exist in the Customers table.

**Result:** 3,296 unique customer IDs were not found in the Customers table.

In [4]:
missing_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]
print(missing_customers.shape)
print(missing_customers["customer_id"].nunique())

(4000, 9)
3296


#### Check 2: Investigation of Missing Customer IDs

**Investigation:** Unmatched orders fall between 1 Jan and May 2024, while customer signup data ranges from 2022 to 1 Jan 2024.

**Possible Reason:** These customers may have signed up after 1 Jan 2024, so their records are missing from the Customers table. This explanation is based on the observed data and is not 100% confirmed.

In [5]:
missing_customers["order_date"].min(), missing_customers["order_date"].max()

('01-01-2024', '31-05-2024')

#### Check 2: Orders–Warehouses Cross-Check

**Objective:** Verify that all warehouse IDs in the Orders table exist in the Warehouses table.
**Result:** No missing warehouse IDs found.

In [6]:
missing_warehouses =orders[
    ~orders["warehouse_id"].isin(warehouses["warehouse_id"])]

print(missing_warehouses.shape)
print(missing_warehouses["warehouse_id"].nunique())

(0, 9)
0


#### Check 3: Orders Items–Orders Cross-Check

**Objective:** Verify that all order IDs in Orders Items exist in Orders.
**Result:** No unmatched order IDs found.

In [7]:
missing_orders = orders_items[
    ~orders_items["order_id"].isin(orders["order_id"])
]

print(missing_orders.shape)
print(missing_orders["order_id"].nunique())

(0, 6)
0


#### Check 4: Orders Items–Products Cross-Check

**Objective:** Verify that all product IDs in Orders Items exist in Products.

**Result:** 5,000 unmatched records found for product ID `P9999`. Possible placeholder ID or dataset generation issue.

In [8]:
unmatched_order_products = orders_items[
    ~orders_items["product_id"].isin(products["product_id"])
]

print(unmatched_order_products.shape[0])
print(unmatched_order_products["product_id"].nunique())
print(unmatched_order_products["product_id"].value_counts())


5000
1
product_id
P9999    5000
Name: count, dtype: int64


In [13]:
# Display the first 10 records with unmatched product IDs to inspect their order details.
unmatched_order_products[
    ["order_id", "product_id", "quantity", "unit_price_at_sale", "line_total"] 
    ].head(10)


,order_id,product_id,quantity,unit_price_at_sale,line_total
639,ORD0000382,P9999,1,1466.35,1466.35
724,ORD0000432,P9999,2,1680.07,3360.14
783,ORD0000468,P9999,2,1546.04,3092.08
1150,ORD0000692,P9999,3,2379.30,7137.90
1677,ORD0001009,P9999,2,1746.16,3492.32
1694,ORD0001018,P9999,2,2873.93,5747.86
2175,ORD0001295,P9999,2,2775.14,5550.28
2616,ORD0001567,P9999,1,1668.23,1668.23
2663,ORD0001594,P9999,1,2577.29,2577.29
3749,ORD0002260,P9999,3,849.50,2548.50


#### Check 5: Deliveries–Orders Cross-Check

**Objective:** Verify that all order IDs in Deliveries exist in Orders.
**Result:** No unmatched order IDs found.

In [14]:
# Find delivery records whose order IDs are not present in the Orders table.
unmatched_deliveries = deliveries[
    ~deliveries["order_id"].isin(orders["order_id"])
]

unmatched_deliveries.shape[0]

0

#### Check 6: Returns–Orders Items Cross-Check

**Objective:** Verify that all order item IDs in Returns exist in Orders Items.
**Result:** No unmatched order item IDs found.

In [15]:
# Find return records whose order_item_id is not present in Orders Items.
unmatched_returns = returns[
    ~returns["order_item_id"].isin(orders_items["order_item_id"])
]

unmatched_returns.shape[0]

0

#### Check 7: Orders Items – Line Total Validation

**Objective:** Verify that line_total equals quantity × unit_price_at_sale.

**Result:** No meaningful calculation mismatches found (within ₹0.01 tolerance).

In [16]:
# Check whether line_total matches quantity multiplied by unit_price_at_sale.
incorrect_line_totals = orders_items[
    orders_items["line_total"]
    != orders_items["quantity"] * orders_items["unit_price_at_sale"]
]

incorrect_line_totals.shape[0]

136638

In [17]:
# Inspect the first 10 records with mismatched line totals.
incorrect_line_totals[
    ["quantity", "unit_price_at_sale", "line_total"]
].head(10)

,quantity,unit_price_at_sale,line_total
10,5,88.82,444.10
13,3,1139.12,3417.36
20,5,337.14,1685.70
66,5,312.84,1564.20
117,3,2666.26,7998.78
122,3,1527.52,4582.56
123,3,3629.93,10889.79
152,5,1218.18,6090.90
154,3,3179.79,9539.37
155,5,2675.28,13376.40


In [18]:
# Check line totals while allowing tiny floating-point differences.
incorrect_line_totals = orders_items[
    ~np.isclose(
        orders_items["line_total"],
        orders_items["quantity"] * orders_items["unit_price_at_sale"],
        atol=0.01
    )
]

incorrect_line_totals.shape[0]

0

#### Product Name & Category Cross-Check

**Finding:** Same product names have different product IDs and prices, possibly due to different brands or variants. Category values have inconsistent capitalization.

**Note:** Brand details are unavailable, so the reason for price differences cannot be confirmed.

In [19]:
# Identify product names that appear with multiple product IDs.
duplicate_product_names = products[
    products["product_name"].duplicated(keep=False)
].sort_values("product_name")

duplicate_product_names[
    ["product_id", "product_name", "category", "sub_category", "unit_price", "unit_cost"]
].head(10)

,product_id,product_name,category,sub_category,unit_price,unit_cost
239,P0240,Charging Cable,Electronics,Mobile Accessories,516.11,291.89
205,P0206,Charging Cable,electronics,Mobile Accessories,2694.43,1515.84
80,P0081,Charging Cable,Electronics,Mobile Accessories,832.57,387.82
286,P0287,Charging Cable,Electronics,Mobile Accessories,430.53,223.58
34,P0035,Charging Cable,electronics,Mobile Accessories,3288.05,1609.27
186,P0187,Charging Cable,Electronics,Mobile Accessories,2765.62,1739.57
24,P0025,Charging Cable,electronics,Mobile Accessories,3333.00,1648.20
27,P0028,Cotton T-Shirt,Apparel,Men's Wear,1813.52,1265.79
72,P0073,Cotton T-Shirt,apparel,Men's Wear,4134.15,1929.31
19,P0020,Cotton T-Shirt,Apparel,Men's Wear,1097.31,520.92


In [20]:
# Check whether the same product name is assigned to multiple categories.
products.groupby("product_name")["category"].nunique().sort_values(ascending=False).head(10)

product_name
Charging Cable    2
Cotton T-Shirt    2
Denim Jeans       2
Face Wash         2
Fitness Band      2
Foundation        2
Hair Oil          2
Kurti             2
Lipstick          2
Moisturizer       2
Name: category, dtype: int64

In [21]:
# Check whether the same product name is assigned to multiple sub-categories.
print(products["category"].unique())

<StringArray>
['Home & Kitchen',    'Electronics',    'electronics',         'beauty',
 'home & kitchen',        'apparel',         'Beauty',        'Apparel']
Length: 8, dtype: str


#### Check 8: Delivery Date Cross-Check

**Objective:** Verify that delivery date is not earlier than ship date.

**Result:** No invalid delivery date sequences found.

In [22]:
# Convert ship_date and delivery_date into datetime format.
deliveries["ship_date"] = pd.to_datetime(
    deliveries["ship_date"], format="%Y-%m-%d"
)

deliveries["delivery_date"] = pd.to_datetime(
    deliveries["delivery_date"], format="%Y-%m-%d"
)

In [23]:
# Find deliveries where delivery date is earlier than ship date.
invalid_delivery_dates = deliveries[
    deliveries["delivery_date"] < deliveries["ship_date"]
]

invalid_delivery_dates.shape[0]


0

#### Check 9: Returns – Return Quantity Validation

**Objective:** Verify that the returned quantity does not exceed the originally ordered quantity.

**Finding:** 15,046 return records have a `return_quantity` greater than the original `quantity`.

**Investigation:** Checked whether the same `order_item_id` had multiple return records. No duplicate return records were found.

**Conclusion:** Multiple return entries do not explain this issue. The exact reason could not be confirmed from the available data. These records will be investigated during the Data Cleaning phase.

In [24]:
# Add original ordered quantity to the Returns table.
returns_with_quantity = returns.merge(
    orders_items[["order_item_id", "quantity"]],
    on="order_item_id",
    how="left"
)

# Find return records where returned quantity exceeds ordered quantity.
invalid_return_quantity = returns_with_quantity[
    returns_with_quantity["return_quantity"]
    > returns_with_quantity["quantity"]
]

invalid_return_quantity.shape[0]

15046

In [25]:
# Inspect the first 10 returns where returned quantity exceeds ordered quantity.
invalid_return_quantity[
    ["order_item_id", "quantity", "return_quantity"]
].head(10)

,order_item_id,quantity,return_quantity
1,OI00255637,1,3
13,OI00709996,1,2
24,OI00616431,1,3
27,OI01467361,1,3
47,OI00033939,1,2
59,OI01662441,3,5
122,OI01422675,2,4
131,OI00305897,2,3
154,OI00461172,1,2
165,OI00817362,2,3


#### Check 10: Returns – Missing Return Reasons

**Objective:** Identify missing values in the `return_reason` column.
**Finding:** 50,063 return records have a missing `return_reason`.

**Conclusion:** Return reasons are unavailable for these records. The missing values will be reviewed during the Data Cleaning phase.

In [26]:
# Count missing values in the return_reason column.
returns["return_reason"].isna().sum()

np.int64(50063)

#### Check 11: Customers – Signup Date vs Order Date

**Objective:** Verify that customers placed orders on or after their signup date.
**Result:** No orders were found before the customer's signup date.

**Conclusion:** No date-sequence inconsistency was found in the matched Orders and Customers records.

In [27]:
# Combine order dates with customer signup dates.
orders_customers = orders.merge(
    customers[["customer_id", "signup_date"]],
    on="customer_id",
    how="inner"
)

In [29]:
# Convert order_date from DD-MM-YYYY format.
orders_customers["order_date"] = pd.to_datetime(
    orders_customers["order_date"],
    format="%d-%m-%Y"
)

# Convert signup_date from YYYY-MM-DD format.
orders_customers["signup_date"] = pd.to_datetime(
    orders_customers["signup_date"],
    format="%Y-%m-%d"
)

In [30]:
# Find orders placed before the customer's signup date.
orders_before_signup = orders_customers[
    orders_customers["order_date"]
    < orders_customers["signup_date"]
]

orders_before_signup.shape[0]

0

#### Check 12: Primary Key Uniqueness

**Objective:** Verify that primary key values are unique across all tables.
**Result:** No duplicate primary key values were found in any of the seven tables.

**Conclusion:** All checked primary key columns maintain uniqueness.

In [32]:
# Check duplicate primary key values in all tables.

primary_keys = {
    "orders": (orders, "order_id"),
    "orders_items": (orders_items, "order_item_id"),
    "deliveries": (deliveries, "order_id"),
    "products": (products, "product_id"),
    "customers": (customers, "customer_id"),
    "returns": (returns, "return_id"),
    "warehouses": (warehouses, "warehouse_id")
}

for table_name, (df, key) in primary_keys.items():
    duplicate_count = df[key].duplicated().sum()
    print(f"{table_name}: {duplicate_count} duplicate {key} values")

orders: 0 duplicate order_id values
orders_items: 0 duplicate order_item_id values
deliveries: 0 duplicate order_id values
products: 0 duplicate product_id values
customers: 0 duplicate customer_id values
returns: 0 duplicate return_id values
warehouses: 0 duplicate warehouse_id values


#### Check 13: Returns – Refund Amount Validation

**Objective:** Compare refund amounts with the calculated value of returned items.

**Finding:** 5,647 records have refunds greater than the calculated item value, while 5,738 records have lower refunds. The remaining 288,019 records match the calculated value.

**Conclusion:** Refund differences were identified, but their exact reasons could not be confirmed from the available data. These records may require further investigation during Data Cleaning.

In [33]:
# Add original item quantity and price to the Returns table.
returns_with_price = returns.merge(
    orders_items[
        ["order_item_id", "quantity", "unit_price_at_sale"]
    ],
    on="order_item_id",
    how="left"
)

In [34]:
# Calculate the expected refund based on returned quantity and unit price.
returns_with_price["expected_refund"] = (
    returns_with_price["return_quantity"]
    * returns_with_price["unit_price_at_sale"]
)

# Find records where refund amount exceeds the expected item value.
excess_refund = returns_with_price[
    returns_with_price["refund_amount"]
    > returns_with_price["expected_refund"]
]

# Find records where refund amount is less than the expected item value.
lower_refund = returns_with_price[
    returns_with_price["refund_amount"]
    < returns_with_price["expected_refund"]
]

# Display the validation results.
print("Refund greater than expected:", excess_refund.shape[0])
print("Refund lower than expected:", lower_refund.shape[0])
print("Refund equal to expected:", (
    returns_with_price["refund_amount"]
    == returns_with_price["expected_refund"]
).sum())

Refund greater than expected: 5647
Refund lower than expected: 5738
Refund equal to expected: 288019


## Data Cleaning

### Orders dataset cleaning

In [3]:
# Clean and standardize the order_status column in the Orders table.
orders["order_status"] = orders["order_status"].str.strip().str.title()

In [4]:
# Convert order_date from DD-MM-YYYY format to datetime.
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    format="%d-%m-%Y" )

In [5]:
print(orders["order_date"].dtype)

datetime64[us]


In [24]:
orders.columns

Index(['order_id', 'customer_id', 'order_date', 'warehouse_id',
       'payment_method', 'order_status'],
      dtype='str')

In [7]:
orders.drop(columns=['Unnamed: 6', 'Unnamed: 7',
       'Unnamed: 8'], inplace=True)

C:\Users\Akash\AppData\Local\Temp\ipykernel_27172\3849159896.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  orders.drop(columns=['Unnamed: 6', 'Unnamed: 7',


### Orders_iteam

In [58]:
orders_items.rename(
    columns={"line_total": "total_price"},
    inplace=True
)

orders_items.columns

C:\Users\Akash\AppData\Local\Temp\ipykernel_27172\1559448035.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  orders_items.rename(


Index(['order_item_id', 'order_id', 'product_id', 'quantity',
       'unit_price_at_sale', 'total_price'],
      dtype='str')

### Deliveries dataset cleaning

In [8]:
deliveries["carrier"].value_counts()

carrier
QuickPost           200326
ShipFast            200275
MetroCourier        199702
BlueDart Express    199532
Name: count, dtype: int64

In [25]:
deliveries["ship_date"] = pd.to_datetime(
    deliveries["ship_date"],
    format="%Y-%m-%d"
)

deliveries["delivery_date"] = pd.to_datetime(
    deliveries["delivery_date"],
    format="%Y-%m-%d"
)

In [26]:
deliveries.head()

,order_id,ship_date,delivery_date,carrier
0,ORD0000001,2024-05-14,2024-05-18,QuickPost
1,ORD0000002,2024-06-08,2024-06-12,QuickPost
2,ORD0000003,2024-01-08,2024-01-14,QuickPost
3,ORD0000004,2024-04-29,2024-05-01,BlueDart Express
4,ORD0000005,2024-04-14,2024-04-19,ShipFast


In [27]:
deliveries[["ship_date", "delivery_date"]].dtypes

ship_date        datetime64[us]
delivery_date    datetime64[us]
dtype: object

### Customrt dataset cleaning

In [ ]:

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    format="%Y-%m-%d"
)

In [32]:
customers["signup_date"].dtype

dtype('<M8[us]')

In [39]:
customers["region"] = (
    customers["region"]
    .str.strip()
    .str.title() )

print(customers["region"].value_counts())

region
North    12616
West     12522
South    12432
East     12430
Name: count, dtype: int64


In [38]:
customers["city "] = (
    customers["city"]
    .str.strip()
    .str.title() )

customers["city"].value_counts()

city
Chandigarh     6340
Delhi          6276
Bhubaneswar    6236
Kolkata        6194
Bangalore      4222
Pune           4214
Ahmedabad      4159
Mumbai         4149
Chennai        4119
Hyderabad      4091
Name: count, dtype: int64

### Products dataset cleaning

In [45]:
products["product_name"] = (
    products["product_name"]
    .str.strip()
    .str.title() )

products["product_name"].value_counts()

product_name
Hair Oil                 19
Wardrobe Organizer       18
Plastic Container Set    17
Kurti                    17
Fitness Band             17
Foundation               17
Smartwatch Pro           16
Face Wash                16
Wall Clock               13
Wireless Earbuds         13
Lipstick                 12
Denim Jeans              12
Cotton T-Shirt           12
Summer Dress             12
Pressure Cooker          11
Table Lamp               11
Over-Ear Headphones      11
Shampoo                  10
Running Shoes             8
Moisturizer               8
Screen Protector          7
Charging Cable            7
Non-Stick Pan             6
Phone Case                5
Sandals                   5
Name: count, dtype: int64

In [43]:
products["category"] = (
    products["category"]
    .str.strip()
    .str.title() )

products["category"].value_counts()

category
Beauty            82
Home & Kitchen    76
Electronics       76
Apparel           66
Name: count, dtype: int64

In [48]:
products["sub_category"] = (
    products["sub_category"]
    .str.strip()
    .str.title() )

products["sub_category"].value_counts()

sub_category
Storage               35
Smartwatches          33
Makeup                29
Women'S Wear          29
Haircare              29
Decor                 24
Men'S Wear            24
Skincare              24
Headphones            24
Mobile Accessories    19
Cookware              17
Footwear              13
Name: count, dtype: int64

In [49]:
products.rename(
    columns={"unit_price": "selling_price"},
    inplace=True
)

C:\Users\Akash\AppData\Local\Temp\ipykernel_27172\995835680.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  products.rename(


### Return dataset cleaning

In [50]:
returns["return_reason"] = (
    returns["return_reason"]
    .str.strip()
    .str.title()
)

print(returns["return_reason"].value_counts())

return_reason
Size Issue          50127
Wrong Item          49935
Not As Described    49830
Damaged             49803
Changed Mind        49646
Name: count, dtype: int64


In [54]:

returns["return_date"] = pd.to_datetime(
    returns["return_date"],
    format="%Y-%m-%d"
)

returns["return_date"].dtype

dtype('<M8[us]')

In [55]:
returns["return_reason"] = returns["return_reason"].fillna("Unknown")

### Warehouses

In [56]:
warehouses["warehouse_name"] = (
    warehouses["warehouse_name"]
    .str.strip()
    .str.title()
)

print(warehouses["warehouse_name"].value_counts())

warehouse_name
North Distribution Center    1
South Distribution Center    1
East Distribution Center     1
West Distribution Center     1
Name: count, dtype: int64


## Cleaned data

In [60]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [61]:
orders.to_csv("../data/processed/orders_cleaned.csv", index=False)

orders_items.to_csv("../data/processed/orders_items_cleaned.csv", index=False)

deliveries.to_csv("../data/processed/deliveries_cleaned.csv", index=False)

customers.to_csv("../data/processed/customers_cleaned.csv", index=False)

products.to_csv("../data/processed/products_cleaned.csv", index=False)

returns.to_csv("../data/processed/returns_cleaned.csv", index=False)

warehouses.to_csv("../data/processed/warehouses_cleaned.csv", index=False)